# Ekstrakcja metadanych produktów (OpenAI)

Dla każdego produktu z `products_zurada.csv` wyciągamy przez LLM:
- `dozwolone_powierzchnie` — lista powierzchni, na których produkt działa poprawnie
- `odradzane_powierzchnie` — lista powierzchni, na których **NIE** należy stosować (krytyczne dla RAG)
- `odczyn_ph` — wartość lub opis odczynu pH
- `metoda_mycia` — lista metod aplikacji (ręczne, maszynowe, ciśnieniowe…)
- `zgodnosc_i_certyfikaty` — certyfikaty i normy (HACCP, EU Ecolabel…)
- `ostrzezenia_bhp` — lista ostrzeżeń BHP z opisów

Wynik trafia do `zurada_products_with_metadata.csv`.

**Checkpoint:** jeśli przerwiesz i uruchomisz ponownie, przetworzone wiersze są pomijane.

In [1]:
# Instalacja zależności (uruchom raz jeśli brakuje pakietów)
%pip install openai pandas  ipywidgets python-dotenv tqdm --quiet

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import json
import time
import pandas as pd
from tqdm.notebook import tqdm
from openai import OpenAI

# ── Konfiguracja ─────────────────────────────────────────────────────────────
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "Brak klucza API")  # ustaw zmienną środowiskową lub wklej tu
MODEL          = "gpt-5.5"                     
INPUT_CSV      = "products_zurada.csv"
CHECKPOINT_CSV = "zurada_metadata_checkpoint.csv"  # plik roboczy (wznawianie)
OUTPUT_CSV     = "zurada_products_with_metadata.csv"
SLEEP_BETWEEN  = 0.3                               # sekund między requestami (rate limit)
MAX_RETRIES    = 3

client = OpenAI(api_key=OPENAI_API_KEY)
print(f"Model: {MODEL}  |  Input: {INPUT_CSV}")

Model: gpt-5.5  |  Input: products_zurada.csv


In [4]:
# ── Wczytanie danych ──────────────────────────────────────────────────────────
df = pd.read_csv(INPUT_CSV)
print(f"Wczytano {len(df)} produktów. Kolumny: {list(df.columns)}")
df.head(2)

Wczytano 100 produktów. Kolumny: ['product_id', 'product_name', 'title', 'content_encoded', 'short_description', 'right_description', 'technical_sheet', 'categories']


,product_id,product_name,title,content_encoded,short_description,right_description,technical_sheet,categories
0,1,Zurada Lśniąca Pianka,Pianka do powierzchni,Smart Foam to aktywna pianka do czyszczenia rę...,Inteligentna aktywna pianka czyszcząco-odtłusz...,Właściwości: Preparat gotowy do użycia. Uniwer...,NaN,"horeca-hotele, szkoly, firmy-sprzatajace, higi..."
1,2,Zurada Szklany Blask,Płyn do szyb i luster,"Środek do ręcznego i natryskowego mycia szyb, ...","Płyn z alkoholem do mycia szyb, luster oraz in...",Właściwości: Preparat gotowy do użycia. Idealn...,gotowy do użycia preparat w formie pianki do c...,"mycie-szyb-i-luster-instytucje, horeca-hotele,..."


In [5]:
# ── Schemat JSON (Structured Outputs) ────────────────────────────────────────
RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "dozwolone_powierzchnie": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Powierzchnie/materiały, na których produkt działa poprawnie."
        },
        "odradzane_powierzchnie": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Powierzchnie/materiały, na których NIE NALEŻY stosować produktu."
        },
        "odczyn_ph": {
            "type": "string",
            "description": "Wartość lub opis pH, np. 'silnie zasadowy', 'neutralny', 'pH > 12.5', 'lekko kwaśny'."
        },
        "metoda_mycia": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Metody aplikacji: reczne, maszynowe, cišnieniowe, pianowe, natryskowe, CIP, zanurzeniowe itp."
        },
        "zgodnosc_i_certyfikaty": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Certyfikaty, normy i zgodności: HACCP, EU Ecolabel, IFRA, RSPO, PN-EN, biodegradowalny itp."
        },
        "ostrzezenia_bhp": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Ostrzeżenia BHP i ograniczenia: nie stosować na rozgrzane powierzchnie, unikać kontaktu z oczami, stosować rękawice itp."
        }
    },
    "required": [
        "dozwolone_powierzchnie",
        "odradzane_powierzchnie",
        "odczyn_ph",
        "metoda_mycia",
        "zgodnosc_i_certyfikaty",
        "ostrzezenia_bhp"
    ],
    "additionalProperties": False
}

SYSTEM_PROMPT = """Jesteś ekspertem ds. chemii czyszczącej. 
Analizujesz opisy produktów czyszczących i wyciągasz ustrukturyzowane metadane.
Odpowiadasz WYŁĄCZNIE poprawnym JSON zgodnym z podanym schematem.
Jeśli informacja nie występuje w tekście, zwróć pustą listę [] lub pusty string \"\".
Nie wymyślaj informacji — wyciągaj tylko to, co jest w tekście."""

print("Schemat i system prompt gotowe.")

Schemat i system prompt gotowe.


In [6]:
# ── Funkcja ekstrakcji dla jednego produktu ───────────────────────────────────
def build_product_text(row: pd.Series) -> str:
    """Łączy pola opisowe produktu w jeden blok tekstu dla LLM."""
    parts = [
        f"Nazwa: {row.get('product_name', '')}",
        f"Tytuł: {row.get('title', '')}",
        f"Opis krótki: {row.get('short_description', '')}",
        f"Opis pełny: {row.get('content_encoded', '')}",
        f"Właściwości: {row.get('right_description', '')}",
        f"Karta techniczna: {row.get('technical_sheet', '')}",
    ]
    return "\n".join(p for p in parts if p.split(": ", 1)[-1].strip())


def extract_metadata(row: pd.Series) -> dict:
    """Wywołuje OpenAI i zwraca słownik z metadanymi. Retries przy błędach."""
    product_text = build_product_text(row)
    user_message = f"""Wyciągnij metadane z poniższego opisu produktu czyszczącego:\n\n{product_text}"""

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_message},
                ],
                response_format={
                    "type": "json_schema",
                    "json_schema": {
                        "name": "product_metadata",
                        "strict": True,
                        "schema": RESPONSE_SCHEMA,
                    }
                },
                max_completion_tokens=800,
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            wait = 2 ** attempt
            print(f"  [RETRY {attempt}/{MAX_RETRIES}] product_id={row['product_id']} — {e} — czekam {wait}s")
            time.sleep(wait)

    # Po wszystkich retry zwróć puste wartości
    print(f"  [FAILED] product_id={row['product_id']} — zwracam puste metadane")
    return {
        "dozwolone_powierzchnie":  [],
        "odradzane_powierzchnie":  [],
        "odczyn_ph":               "",
        "metoda_mycia":            [],
        "zgodnosc_i_certyfikaty":  [],
        "ostrzezenia_bhp":         [],
    }


print("Funkcja extract_metadata gotowa.")

Funkcja extract_metadata gotowa.


In [7]:
# ── Test na jednym produkcie (opcjonalne, usuń # żeby sprawdzić) ────────────
sample = df.iloc[0]
result = extract_metadata(sample)
print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "dozwolone_powierzchnie": [
    "wszystkie rodzaje powierzchni",
    "plastik",
    "meble",
    "blaty",
    "okapy",
    "elementy metalowe",
    "powierzchnie mające kontakt z żywnością",
    "szkło"
  ],
  "metoda_mycia": [
    "ręczne",
    "natryskowe"
  ],
  "odczyn_ph": "",
  "odradzane_powierzchnie": [],
  "ostrzezenia_bhp": [
    "Po zastosowaniu na powierzchniach mających kontakt z żywnością należy spłukać je czystą wodą pitną.",
    "Na szkle może pozostawiać smugi, dlatego po użyciu zaleca się przetarcie powierzchni do sucha."
  ],
  "zgodnosc_i_certyfikaty": []
}


In [8]:
# ── Główna pętla z checkpointem ───────────────────────────────────────────────
METADATA_COLS = [
    "dozwolone_powierzchnie",
    "odradzane_powierzchnie",
    "odczyn_ph",
    "metoda_mycia",
    "zgodnosc_i_certyfikaty",
    "ostrzezenia_bhp",
]

# Wczytaj checkpoint jeśli istnieje
if os.path.exists(CHECKPOINT_CSV):
    done_df = pd.read_csv(CHECKPOINT_CSV)
    done_ids = set(done_df["product_id"].tolist())
    print(f"Checkpoint: {len(done_ids)} produktów już przetworzonych.")
else:
    done_df = pd.DataFrame()
    done_ids = set()
    print("Brak checkpointu — zaczynam od początku.")

results = []
to_process = df[~df["product_id"].isin(done_ids)]
print(f"Do przetworzenia: {len(to_process)} produktów.")

for _, row in tqdm(to_process.iterrows(), total=len(to_process), desc="Ekstrakcja metadanych"):
    meta = extract_metadata(row)

    # Zapisz listy jako JSON string (CSV-friendly)
    record = {"product_id": row["product_id"]}
    for col in METADATA_COLS:
        val = meta.get(col, [] if col != "odczyn_ph" else "")
        record[col] = json.dumps(val, ensure_ascii=False) if isinstance(val, list) else val

    results.append(record)
    time.sleep(SLEEP_BETWEEN)

    # Zapis checkpointu co 5 produktów
    if len(results) % 5 == 0:
        batch_df = pd.DataFrame(results)
        combined = pd.concat([done_df, batch_df], ignore_index=True) if not done_df.empty else batch_df
        combined.to_csv(CHECKPOINT_CSV, index=False)

# Finalny checkpoint
if results:
    batch_df = pd.DataFrame(results)
    done_df = pd.concat([done_df, batch_df], ignore_index=True) if not done_df.empty else batch_df
    done_df.to_csv(CHECKPOINT_CSV, index=False)

print(f"\nGotowe. Łącznie w checkpoincie: {len(done_df)} produktów.")

Brak checkpointu — zaczynam od początku.
Do przetworzenia: 100 produktów.


Ekstrakcja metadanych:   0%|          | 0/100 [00:00<?, ?it/s]

  [RETRY 1/3] product_id=15 — Expecting value: line 1 column 1 (char 0) — czekam 2s

Gotowe. Łącznie w checkpoincie: 100 produktów.


In [9]:
# ── Scalenie z oryginalnym CSV i zapis outputu ────────────────────────────────
metadata_df = pd.read_csv(CHECKPOINT_CSV)

# Upewnij się, że typy product_id są zgodne
df["product_id"] = df["product_id"].astype(int)
metadata_df["product_id"] = metadata_df["product_id"].astype(int)

merged = df.merge(metadata_df, on="product_id", how="left")
merged.to_csv(OUTPUT_CSV, index=False)

print(f"Zapisano {len(merged)} wierszy do '{OUTPUT_CSV}'")
print(f"Kolumny: {list(merged.columns)}")
merged[["product_id", "product_name"] + METADATA_COLS].head(5)

Zapisano 100 wierszy do 'zurada_products_with_metadata.csv'
Kolumny: ['product_id', 'product_name', 'title', 'content_encoded', 'short_description', 'right_description', 'technical_sheet', 'categories', 'dozwolone_powierzchnie', 'odradzane_powierzchnie', 'odczyn_ph', 'metoda_mycia', 'zgodnosc_i_certyfikaty', 'ostrzezenia_bhp']


,product_id,product_name,dozwolone_powierzchnie,odradzane_powierzchnie,odczyn_ph,metoda_mycia,zgodnosc_i_certyfikaty,ostrzezenia_bhp
0,1,Zurada Lśniąca Pianka,"[""wszystkie rodzaje powierzchni"", ""plastik"", ""...",[],NaN,"[""ręczne"", ""natryskowe""]",[],"[""Po zastosowaniu na powierzchniach mających k..."
1,2,Zurada Szklany Blask,"[""szyby"", ""lustra"", ""inne szklane elementy"", ""...","[""powierzchnie wrażliwe na działanie alkoholu""...",NaN,"[""ręczne"", ""natryskowe"", ""pianowe""]","[""kompozycja zapachowa zgodna ze standardami I...","[""Nie zaleca się stosowania na powierzchniach ..."
2,3,Zurada Tłuszcz Stop,"[""blaty kuchenne"", ""podłogi"", ""ściany"", ""naczy...","[""aluminium"", ""miedź"", ""powierzchnie malowane""...",NaN,"[""pianowe"", ""ręczne""]",[],"[""Przed użyciem należy dobrać stężenie do stop..."
3,4,Zurada Kuchnia Czysta,"[""powierzchnie kuchenne"", ""urządzenia kuchenne...",[],NaN,[],"[""kompozycja zapachowa zgodna ze standardami I...",[]
4,5,Zurada Piana Moc,"[""grille"", ""piekarniki"", ""ruszty"", ""piece"", ""u...","[""aluminium"", ""miedź"", ""powierzchnie malowane""...",NaN,"[""pianowe"", ""ręczne"", ""natryskowe""]","[""HACCP"", ""RSPO""]",[]


In [10]:
# ── Walidacja: produkty z pustymi odradzane_powierzchnie ─────────────────────
def is_empty(val):
    if pd.isna(val):
        return True
    try:
        parsed = json.loads(val)
        return len(parsed) == 0
    except Exception:
        return str(val).strip() == ""

mask = merged["odradzane_powierzchnie"].apply(is_empty)
print(f"Produkty BEZ odradzanych powierzchni ({mask.sum()} szt.):")
merged.loc[mask, ["product_id", "product_name", "odradzane_powierzchnie"]]

Produkty BEZ odradzanych powierzchni (46 szt.):


,product_id,product_name,odradzane_powierzchnie
0,1,Zurada Lśniąca Pianka,[]
3,4,Zurada Kuchnia Czysta,[]
6,7,Zurada Naczynia Generic Cleaning,[]
10,11,Zurada Czysta Łazienka,[]
12,13,Zurada Total Clean,[]
17,18,Zurada Turbo Czystość,[]
19,20,Zurada Nabłyszcz Pro,[]
20,21,Zurada Krystal Błysk,[]
22,23,Zurada Doza,[]
23,24,Zurada Czysta Posadzka,[]


In [11]:
# ── Walidacja: rozkład certyfikatów ──────────────────────────────────────────
from collections import Counter

cert_counter = Counter()
for val in merged["zgodnosc_i_certyfikaty"].dropna():
    try:
        certs = json.loads(val)
        cert_counter.update(certs)
    except Exception:
        pass

print("Top certyfikaty w całym katalogu:")
for cert, count in cert_counter.most_common(20):
    print(f"  {count:3d}x  {cert}")

Top certyfikaty w całym katalogu:
   24x  RSPO
   20x  IFRA
   17x  certyfikowany olej palmowy zgodny z RSPO
   11x  kompozycja zapachowa zgodna ze standardami IFRA
   10x  HACCP
    4x  ekologiczny
    3x  nie zawiera fosforanów
    2x  zgodność kompozycji zapachowej ze standardami IFRA
    2x  nie zawiera SLS
    2x  nie zawiera dodecylobenzenosulfonianów
    2x  nie zawiera EDTA
    2x  nie zawiera NTA
    1x  nie zawiera EDTA, NTA, SLS, fosforanów ani dodecylobenzenosulfonianów
    1x  bezfosforanowy
    1x  nie obciąża środowiska
    1x  nie powoduje zarastania glonami rzek i mórz
    1x  opakowanie nadaje się do ponownego napełnienia
    1x  elementy opakowania można poddać recyklingowi
    1x  składniki pochodzenia roślinnego
    1x  europejskie normy mikrobiologiczne


In [12]:
# ── Walidacja: rozkład metod mycia ───────────────────────────────────────────
method_counter = Counter()
for val in merged["metoda_mycia"].dropna():
    try:
        methods = json.loads(val)
        method_counter.update(methods)
    except Exception:
        pass

print("Metody mycia w katalogu:")
for method, count in method_counter.most_common():
    print(f"  {count:3d}x  {method}")

Metody mycia w katalogu:
   78x  ręczne
   39x  pianowe
   25x  maszynowe
   21x  ciśnieniowe
   17x  natryskowe
    5x  CIP
    4x  wysokociśnieniowe
    2x  zmywarki gastronomiczne
    2x  zmywarki przemysłowe
    2x  systemy dozujące
    2x  myjnie samochodowe
    2x  spryskiwacz
    2x  automatyczne
    2x  aplikacja gąbką
    1x  szorowanie
    1x  zanurzeniowe
    1x  wybielanie
    1x  mycie w zmywarkach
    1x  dozowanie przez dozownik
    1x  z systemem dozującym
    1x  dozowanie systemami dozującymi
    1x  piankowe
    1x  przecieranie ściereczką
    1x  przecieranie mopem
    1x  automatyczne mycie pieców
    1x  mycie ręczne
    1x  mycie ciśnieniowe
    1x  mycie natryskowe
    1x  ręczne nakładanie
    1x  aplikacja mopem
    1x  systemy automatycznego dozowania
    1x  roboty myjące
    1x  mopy elektryczne
    1x  rozpylić w pomieszczeniu
    1x  atomizer
    1x  automaty szorująco-zbierające
    1x  wcieranie w dłonie
    1x  myjki tunelowe
    1x  automatyczne płuka

In [13]:
# ── Podgląd konkretnego produktu ─────────────────────────────────────────────
PRODUCT_ID = 45  # zmień na dowolny ID do inspekcji

row = merged[merged["product_id"] == PRODUCT_ID].iloc[0]
print(f"=== {row['product_name']} (ID {PRODUCT_ID}) ===")
for col in METADATA_COLS:
    val = row[col]
    try:
        val = json.loads(val)
    except Exception:
        pass
    print(f"\n{col}:")
    if isinstance(val, list):
        for item in val:
            print(f"  • {item}")
    else:
        print(f"  {val}")

=== Zurada Odtłuszczacz Moc (ID 45) ===

dozwolone_powierzchnie:
  • blaty
  • podłogi
  • ściany
  • naczynia
  • narzędzia warsztatowe
  • części samochodowe
  • silnie zatłuszczone powierzchnie
  • sufity
  • przedmioty
  • powierzchnie wrażliwe na działanie alkaliów
  • aluminium
  • powierzchnie mające kontakt z żywnością

odradzane_powierzchnie:

odczyn_ph:
  nan

metoda_mycia:
  • ręczne
  • ciśnieniowe
  • maszynowe
  • CIP

zgodnosc_i_certyfikaty:
  • Można stosować w przemyśle spożywczym

ostrzezenia_bhp:
  • Po użyciu na powierzchniach mających kontakt z żywnością należy spłukać czystą wodą pitną.
  • Przed pierwszym zastosowaniem wykonać próbę w niewidocznym miejscu.
